# 🔍 Explore a real dataset

Meet **EU Media Habits 2026** — a survey of 624 people across 8 European countries about how they consume news and media. *(The data is synthetic — generated for teaching — but shaped like a real survey.)*

This is **Exploratory Data Analysis (EDA)**: the first thing a data scientist does with any new dataset — look at it, summarise it, visualise it, question it.

## Step 1 — Load the data

`pandas` is Python's workhorse for tabular data:

In [ ]:
import pandas as pd

df = pd.read_csv("data/eu_media_habits.csv")
df.head()  # first 5 rows

Each **row** is one survey respondent; each **column** is one answer. How big is the table, and what types are the columns?

In [ ]:
print("Rows and columns:", df.shape)
df.info()

Notice several columns have **missing values** (fewer non-null entries than rows) — real data is never perfectly clean. Track 1's *[Seeing what's missing]* lesson showed you how to diagnose those gaps; here we deal with them.

## Step 2 — Summarise

`describe()` gives instant statistics for every numeric column:

In [ ]:
df.describe().round(2)

Read that table for a moment. What is the average age? The maximum daily social media hours? Summary tables are the fastest reality-check in data science.

### Dealing with the missing values

Count them, then fill them with the column median (a common, defensible choice):

In [ ]:
print(df.isna().sum())

df["hours_tv"] = df["hours_tv"].fillna(df["hours_tv"].median())
print("\nMissing values in hours_tv after filling:", df["hours_tv"].isna().sum())

### Categorical columns

For text columns, `value_counts()` shows the distribution:

In [ ]:
df["main_news_source"].value_counts()

## Step 3 — Compare groups

`groupby` is EDA's power tool: split the data by a category, then aggregate. Which countries spend most time on social media?

In [ ]:
social_by_country = df.groupby("country")["hours_social"].mean().sort_values(ascending=False)
social_by_country.round(2)

## Step 4 — Visualise

Numbers persuade analysts; pictures persuade everyone.

In [ ]:
import matplotlib.pyplot as plt

social_by_country.plot(kind="barh", color="#1d3fbf")
plt.xlabel("Average hours of social media per day")
plt.title("Social media use by country — EU Media Habits 2026")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
df["age"].plot(kind="hist", bins=25, color="#1d3fbf", edgecolor="white")
plt.xlabel("Age")
plt.title("Age distribution of respondents")
plt.show()

### Relationships between variables

Does heavy social media use go together with trust in the news? A scatter plot and a correlation matrix answer this:

In [ ]:
df.plot(kind="scatter", x="hours_social", y="trust_news", alpha=0.35, color="#1d3fbf")
plt.title("Social media hours vs. trust in news")
plt.show()

In [ ]:
df[["age", "hours_tv", "hours_social", "hours_news", "trust_news"]].corr().round(2)

A correlation near **0** means no linear relationship; near **+1/−1** means a strong one. Look at `hours_social` vs `trust_news`: there is a clear **negative** correlation in this data. ⚠️ **Correlation is not causation** — and the stronger the correlation, the more tempting the causal story becomes. Does social media erode trust? Do people who already distrust the news turn to social media instead? Is something else driving both? This data cannot tell you.

## ⚖️ Ethics lab — before it becomes a headline

You have just produced the kind of result that ends up in a news story. Look at your country ranking and imagine the headline: *"Austrians spend the least time on social media in Europe."* This lab asks whether the data can actually carry that sentence.

**Beat 1 — Build it** ✅ (the country ranking above). Now:

**Beat 2 — Measure it.** An average is only half a fact. The other half is *how sure you can be of it* — which depends on how many people it rests on and how much they vary. `compare_groups()` puts an approximate 95% confidence interval around every group's average:

In [ ]:
%run datalab_ethics.py

compare_groups(df, "hours_social", by="country")

Read the chart from the top. **Portugal's interval stands clear of everyone else's** — that difference is real. Below it, every neighbouring pair overlaps, all the way down to the bottom: seven countries put in order by differences smaller than the margin of error. "Austria last" is a statistical tie with Germany.

Now the second question every responsible analyst asks: *how many people does each number rest on?*

In [ ]:
compare_groups(df, "trust_news", by="gender")

**Beat 3 — Name it.** Two different problems, both common in published statistics:

1. **A ranking claims more than the data holds.** Only one step of the country ranking is larger than the uncertainty. The *European Statistics Code of Practice* asks official statistics for *accuracy and reliability* and for *clarity* — which means publishing the uncertainty alongside the average, not quietly dropping it. And a country average says nothing about any single person in that country (the *ecological fallacy*).
2. **Some groups are almost invisible.** About a dozen respondents chose `other` for gender, and fewer than ten of them answered the trust question — so their interval is over three times wider than everyone else's. Split by country, that is between one and three people per cell: a figure for "respondents who chose *other* in Iceland" describes **one identifiable person**. Under the GDPR (Recital 26), data is only anonymous if no one can be *singled out*.

There is no clean answer to the second problem. Drop the group, and you erase people who are already under-counted. Report them on their own, and you publish noise — and possibly a person. The professional middle ground: report them **with the uncertainty stated**, never break them down further, and say plainly when a sample is too small to support a claim.

**Beat 4 — Judge it.** Your call, in writing:

In [ ]:
my_decision = """
Your answer: write the headline you would actually publish from these country
results, and one sentence on how you would report the respondents who chose 'other'.
"""

reflect_submit("eda-headline-ethics", my_decision)

## ✅ What you learned

The complete first-contact EDA workflow: `read_csv` → `head` / `info` / `describe` → handle missing values → `value_counts` → `groupby` → plots → correlations — and the step most analyses skip: **measuring how certain each group's number is** before anyone turns it into a headline.

📚 **Go further:**
- [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html) — the official quick tour
- [Python for Data Analysis](https://wesmckinney.com/book/) — the free book by pandas' creator

**Next:** prove it. The *Graded exercise* notebook asks you six questions about this same dataset — with an instant checker.